# 07 — Raw Hazard EDA v2: Case-Study Selection

**Goal:** select a real RASFF case-study for the Regulatory Agent benchmark by starting from the original `hazards` field rather than the derived `Hazard_Type`.

### Analysis flow
`Origin × Category → inspect data completeness → screen pairs → inspect raw hazard strings → split multi-hazard records → light normalization → review aliases → Origin × Category × individual Hazard → time/seriousness → shortlist`

> RASFF notification counts describe reported notifications, not prevalence, exposure, or causal risk.

In [ ]:
from pathlib import Path
import sys, re
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'rasff_core.py').exists():
    raise FileNotFoundError('Open this notebook from the RASFF_Rebuild folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rasff_core import load_prepared
df = load_prepared()

required = ['reference','date','origin','category','hazards','risk_decision','type']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

print(f'Rows: {len(df):,}')
print(f'Date range: {df.date.min()} → {df.date.max()}')
display(df[['reference','date','origin','category','hazards','risk_decision']].head())

## 1. Scope and raw-hazard quality

Start with food notifications with a known origin. Keep the original `hazards` field intact.

In [ ]:
eda = df.loc[(df['type'].str.lower() == 'food') & df['origin'].ne('')].copy()
eda['year'] = eda['date'].dt.year
eda['hazard_raw'] = eda['hazards'].fillna('').astype(str).str.strip()
eda['hazard_missing'] = eda['hazard_raw'].eq('')

print(f'Food rows with known origin: {len(eda):,}')
print(f'Unique origins: {eda.origin.nunique():,}')
print(f'Unique categories: {eda.category.nunique():,}')
print(f'Missing raw hazards: {eda.hazard_missing.sum():,} ({eda.hazard_missing.mean():.1%})')
print(f'Unique non-empty raw hazard strings: {eda.loc[~eda.hazard_missing, "hazard_raw"].nunique():,}')

## 2. Q1 — Which origin × category pairs are repeatedly reported?

First count notifications by `origin × category`. This is a support screen only; high counts do not mean a product/country is more unsafe.

In [ ]:
origin_category = (
    eda.groupby(['origin','category'])
       .agg(
           notifications=('reference','nunique'),
           years=('year','nunique'),
           first_date=('date','min'),
           last_date=('date','max'),
           hazard_missing_share=('hazard_missing','mean'),
           serious_share=('target','mean'),
       )
       .reset_index()
       .sort_values(['notifications','years'], ascending=False)
)

display(origin_category.head(30))

### Before filtering — inspect the top origin × category pairs

Before applying thresholds, inspect the top pairs together with hazard completeness. At this stage, **do not exclude anything yet**.

Useful columns:
- `notifications`: unique RASFF notifications
- `years`: number of calendar years represented
- `hazard_available_pct`: proportion with non-empty raw hazard information
- `hazard_missing_pct`: proportion without raw hazard information
- `serious_pct`: proportion carrying the recorded `serious` label

In [ ]:
TOP_N_PAIRS = 30
pair_overview = origin_category.head(TOP_N_PAIRS).copy()
pair_overview['hazard_available_pct'] = ((1 - pair_overview['hazard_missing_share']) * 100).round(1)
pair_overview['hazard_missing_pct'] = (pair_overview['hazard_missing_share'] * 100).round(1)
pair_overview['serious_pct'] = (pair_overview['serious_share'] * 100).round(1)

display(pair_overview[[
    'origin','category','notifications','years',
    'hazard_available_pct','hazard_missing_pct','serious_pct',
    'first_date','last_date'
]])

## 3. Q2 — Which origin × category pairs have enough hazard information to inspect?

Only after inspecting the overview do we apply editable screening rules. These thresholds reduce the number of pairs to review; they are **not scientific cut-offs**.

In [ ]:
MIN_PAIR_NOTIFICATIONS = 30
MIN_YEARS = 3
MAX_HAZARD_MISSING_SHARE = 0.50

pair_candidates = origin_category.loc[
    (origin_category.notifications >= MIN_PAIR_NOTIFICATIONS) &
    (origin_category.years >= MIN_YEARS) &
    (origin_category.hazard_missing_share <= MAX_HAZARD_MISSING_SHARE)
].copy()

display(pair_candidates.head(30))
print(f'Eligible origin × category pairs: {len(pair_candidates)}')

## 4. Q3 — Inspect raw hazards before normalization

The default selects the highest-count eligible pair. Inspect whether the same hazard appears under multiple spellings and whether one notification contains multiple hazards.

In [ ]:
SELECTED_ORIGIN = pair_candidates.iloc[0]['origin'] if len(pair_candidates) else None
SELECTED_CATEGORY = pair_candidates.iloc[0]['category'] if len(pair_candidates) else None

pair = eda.loc[(eda.origin == SELECTED_ORIGIN) & (eda.category == SELECTED_CATEGORY)].copy()

print('Selected:', SELECTED_ORIGIN, '|', SELECTED_CATEGORY)
print(f'Notifications: {pair.reference.nunique():,}')
print(f'Missing hazards: {pair.hazard_missing.mean():.1%}')

raw_counts = (
    pair.loc[~pair.hazard_missing]
        .groupby('hazard_raw')['reference'].nunique()
        .sort_values(ascending=False)
        .rename('notifications')
)
display(raw_counts.head(40))

display(pair.loc[~pair.hazard_missing,
    ['reference','date','hazard_raw','subject','risk_decision']]
    .sort_values('date', ascending=False).head(30))

## 5. Q4 — Split and normalize individual hazards

The raw `hazards` field may contain several hazards in one notification, for example:

`Aflatoxin B1 ,aflatoxin total ,ochratoxin A`

Treating this as one unique string hides the individual hazards. For EDA we therefore:

1. preserve `hazard_raw`,
2. split comma-separated entries,
3. explode to one `notification × hazard item` row,
4. apply only light formatting normalization,
5. inspect variants before making semantic aliases.

A notification may contain multiple hazards, so hazard counts are **not additive** to the total number of notifications.

In [ ]:
hazard_long = eda.loc[~eda['hazard_missing']].copy()

# The supplied snapshot commonly stores multiple hazards separated by commas.
hazard_long['hazard_list'] = hazard_long['hazard_raw'].str.split(r'\s*,\s*', regex=True)
hazard_long = hazard_long.explode('hazard_list')
hazard_long['hazard_item_raw'] = hazard_long['hazard_list'].fillna('').astype(str).str.strip()
hazard_long = hazard_long.loc[hazard_long['hazard_item_raw'].ne('')].copy()

print('Original notifications with hazard information:',
      f"{eda.loc[~eda.hazard_missing, 'reference'].nunique():,}")
print('Notification × hazard rows after exploding:', f'{len(hazard_long):,}')

display(hazard_long[['reference','origin','category','hazard_raw','hazard_item_raw']].head(20))

In [ ]:
def normalize_hazard_item(value):
    """Formatting cleanup only; does not merge scientifically different hazards."""
    if pd.isna(value):
        return ''
    s = str(value).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = s.strip(' ,;:')
    return s

hazard_long['hazard_clean'] = hazard_long['hazard_item_raw'].map(normalize_hazard_item)

print('Unique hazard items before cleaning:', hazard_long['hazard_item_raw'].nunique())
print('Unique hazard items after light cleaning:', hazard_long['hazard_clean'].nunique())

### Inspect individual hazards within the selected pair

This is the first table that directly answers: **within this origin × category pair, which individual hazards are most frequently reported?**

In [ ]:
pair_hazards = hazard_long.loc[
    (hazard_long['origin'] == SELECTED_ORIGIN) &
    (hazard_long['category'] == SELECTED_CATEGORY)
].copy()

hazard_counts = (
    pair_hazards.groupby('hazard_clean')['reference']
                .nunique()
                .sort_values(ascending=False)
                .rename('notifications')
                .reset_index()
)

display(hazard_counts.head(40))

### Review variants before merging

Use a keyword to inspect whether one hazard is fragmented across spelling variants. This helper **does not merge anything**. The review runs across all food notifications so the alias decision is based on the dataset's overall recording pattern, not only one selected pair.

In [ ]:
KEYWORD = 'aflatoxin'  # change to e.g. salmonella, chlorpyrifos, ochratoxin

keyword_review = (
    hazard_long.loc[
        hazard_long['hazard_clean'].str.contains(KEYWORD, case=False, na=False)
    ]
    .groupby('hazard_clean')['reference']
    .nunique()
    .sort_values(ascending=False)
    .rename('notifications')
    .reset_index()
)

display(keyword_review.head(50))

### Conservative alias mapping

Based on the current aflatoxin review, only obvious singular/plural recording variants are merged:

- `aflatoxins b1` → `aflatoxin b1`
- `aflatoxins` → `aflatoxin`

`aflatoxin b1`, `aflatoxin total`, and unspecified `aflatoxin` remain separate because they do not carry identical information.

Add future aliases only after reviewing the raw strings.

In [ ]:
HAZARD_ALIASES = {
    'aflatoxins b1': 'aflatoxin b1',
    'aflatoxins': 'aflatoxin',
}

hazard_long['hazard_normalized'] = hazard_long['hazard_clean'].replace(HAZARD_ALIASES)

aflatoxin_review = (
    hazard_long.loc[
        hazard_long['hazard_normalized'].str.contains('aflatoxin', case=False, na=False)
    ]
    .groupby('hazard_normalized')['reference']
    .nunique()
    .sort_values(ascending=False)
    .rename('notifications')
    .reset_index()
)

display(aflatoxin_review)

## 6. Q5 — Rank origin × category × individual normalized hazard patterns

Now combine the three dimensions using the exploded hazard table. Counts use unique `reference` values so the same notification is not counted twice for the same normalized hazard.

In [ ]:
three_way = (
    hazard_long.groupby(['origin','category','hazard_normalized'])
    .agg(
        notifications=('reference','nunique'),
        years=('year','nunique'),
        first_date=('date','min'),
        last_date=('date','max'),
        serious_share=('target','mean'),
    )
    .reset_index()
)

pair_meta = origin_category[['origin','category','notifications','hazard_missing_share']].rename(
    columns={'notifications':'origin_category_total'}
)
three_way = three_way.merge(pair_meta, on=['origin','category'], how='left')
three_way['share_within_origin_category'] = (
    three_way['notifications'] / three_way['origin_category_total']
)
three_way = three_way.sort_values(
    ['notifications','years','share_within_origin_category'], ascending=False
)

display(three_way.head(40))

print('\nSelected pair — normalized individual hazards')
display(
    three_way.loc[
        (three_way.origin == SELECTED_ORIGIN) &
        (three_way.category == SELECTED_CATEGORY)
    ].head(30)
)

## 7. Q6 — Screen repeated individual-hazard candidates

These are editable benchmark-discovery thresholds, not epidemiological cut-offs.

In [ ]:
MIN_HAZARD_NOTIFICATIONS = 15
MIN_HAZARD_YEARS = 3
MIN_WITHIN_PAIR_SHARE = 0.10

hazard_candidates = three_way.loc[
    (three_way.notifications >= MIN_HAZARD_NOTIFICATIONS) &
    (three_way.years >= MIN_HAZARD_YEARS) &
    (three_way.share_within_origin_category >= MIN_WITHIN_PAIR_SHARE)
].copy()

display(hazard_candidates.head(30))
print(f'Individual-hazard candidate patterns: {len(hazard_candidates)}')

## 8. Q7 — Is a selected pattern persistent over time?

Choose a candidate explicitly. The default uses the highest-count candidate overall; edit the three variables if another case is more scientifically useful.

In [ ]:
if len(hazard_candidates):
    default_case = hazard_candidates.iloc[0]
    CASE_ORIGIN = default_case['origin']
    CASE_CATEGORY = default_case['category']
    CASE_HAZARD = default_case['hazard_normalized']
else:
    CASE_ORIGIN = CASE_CATEGORY = CASE_HAZARD = None

case = hazard_long.loc[
    (hazard_long.origin == CASE_ORIGIN) &
    (hazard_long.category == CASE_CATEGORY) &
    (hazard_long.hazard_normalized == CASE_HAZARD)
].copy()

print('Case:', CASE_ORIGIN, '|', CASE_CATEGORY, '|', CASE_HAZARD)

yearly = case.groupby('year')['reference'].nunique().rename('notifications').reset_index()
display(yearly)

if len(yearly):
    ax = yearly.plot(x='year', y='notifications', kind='bar', legend=False,
                     figsize=(8,4), title='Selected hazard pattern by year')
    ax.set_ylabel('RASFF notifications')
    plt.tight_layout()
    plt.show()

## 9. Q8 — Bring `Hazard_Type` back only as a cross-check

`Hazard_Type` was useful in Project 002 for modeling/visualization, but it should not decide the exact hazard case-study. Here it is only descriptive context.

In [ ]:
if 'Hazard_Type' in case.columns:
    display(
        case.groupby('Hazard_Type')['reference'].nunique()
            .sort_values(ascending=False)
            .rename('notifications')
    )
else:
    print('Hazard_Type is not present — no cross-check needed.')

## 10. Q9 — Build a shortlist for Regulatory Agent feasibility checking

Do **not** automatically choose the final benchmark case from count alone. Review the top candidates for:

- data support and years,
- raw hazard completeness,
- hazard specificity and normalization quality,
- identifiable EU legislation / official guidance,
- usefulness for RAG,
- usefulness for RASFF search/data-analysis tools,
- potential for multi-step investigation,
- interpretability without implying notification count = underlying risk.

The final 2–3 cases should be checked against official regulatory sources before freezing the 10-question benchmark.

In [ ]:
shortlist = hazard_candidates.head(20).copy()

# Descriptive cross-check only
if 'Hazard_Type' in hazard_long.columns:
    type_mode = (
        hazard_long.groupby(['origin','category','hazard_normalized'])['Hazard_Type']
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else '')
        .rename('derived_hazard_type')
        .reset_index()
    )
    shortlist = shortlist.merge(type_mode, on=['origin','category','hazard_normalized'], how='left')

shortlist['raw_hazard_reviewed'] = False
shortlist['regulatory_link_checked'] = False
shortlist['notes'] = ''

display(shortlist)

out = ROOT / 'artifacts' / 'regulatory_agent_individual_hazard_candidates.csv'
shortlist.to_csv(out, index=False)
print('Saved:', out)

## Interpretation guardrails

1. **Notification count ≠ prevalence or true risk.** RASFF records notified events, not all products, imports, inspections or exposures.
2. **Origin ≠ causation.** Trade volume, border-control intensity, targeted sampling and reporting practice can affect counts.
3. **Raw `hazards` comes first.** `Hazard_Type` is used only after candidate discovery.
4. **Multi-hazard notifications are exploded for analysis.** One notification can therefore contribute to more than one hazard.
5. **Counts use unique `reference` within each hazard** to avoid duplicate counting of the same notification/hazard combination.
6. **Normalization is conservative and auditable.** Scientifically different hazards are not merged merely for convenience.
7. **Missing hazards remain a data-quality limitation.** Blank does not mean no hazard existed.
8. **This notebook selects benchmark cases; it does not estimate causal or population-level food-safety risk.**

### Next step
Run the notebook and review:
- `pair_overview`
- `pair_candidates`
- `hazard_counts` for the selected pair
- `keyword_review` for important hazard families
- `three_way`
- `hazard_candidates`

Then verify 2–3 promising cases against official EU regulatory evidence before designing the 10-question Prompt → RAG → Tools → Agent benchmark.